# CPC + Return Regression Training

Two-phase training for crypto trading with **self-supervised pretraining**:

1. **Phase 1: CPC Pretraining** - Self-supervised learning on price sequences (InfoNCE loss)
2. **Phase 2: Return Regression** - Multi-task prediction (return + drawdown) with Gaussian NLL

Uses **Kelly Criterion** for optimal position sizing at inference.

## Why This Approach?

| Problem with RL | Solution |
|-----------------|----------|
| 70% SELL class imbalance | Regression eliminates class imbalance |
| Complex reward engineering | Direct return prediction, no reward shaping |
| Policy collapse | Self-supervised pretraining learns robust features |
| No uncertainty quantification | Gaussian output provides confidence |
| Arbitrary position sizing | Kelly criterion is mathematically optimal |

## Expected Results:
- **Phase 1**: ~1-2h on A100, ~3-4h on T4
- **Phase 2**: ~30-60min
- **Output**: Return predictions with calibrated uncertainty

## GitHub Repository Setup

In [ ]:
# GitHub Configuration
GITHUB_USERNAME = "tr4m0ryp"  # Replace with your GitHub username
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"  # Replace with your GitHub Personal Access Token
GITHUB_REPO_URL = "https://github.com/tr4m0ryp/GMGN_TradingBot.git"

print("GitHub credentials configured")
print(f"Username: {GITHUB_USERNAME}")
print(f"Repository: {GITHUB_REPO_URL}")

In [ ]:
import os
import subprocess

os.chdir('/content')

repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
repo_path = f'/content/{repo_name}'

if os.path.exists(repo_path):
    print(f"Repository '{repo_name}' already exists at {repo_path}")
    print("Pulling latest changes...")
    os.chdir(repo_path)
    subprocess.run(['git', 'pull'], capture_output=True, text=True)
else:
    clone_url = GITHUB_REPO_URL.replace('https://', f'https://{GITHUB_USERNAME}:{GITHUB_TOKEN}@')
    result = subprocess.run(['git', 'clone', clone_url], capture_output=True, text=True, check=True)
    print(f"Successfully cloned repository to {repo_path}")

import sys
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

os.chdir(repo_path)
print(f"Current directory: {os.getcwd()}")

## Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch numpy pandas tqdm matplotlib scikit-learn tensorboard

# Verify installation
import torch
print(f"[OK] PyTorch {torch.__version__}")

try:
    from sklearn.manifold import TSNE
    print("[OK] scikit-learn available")
except ImportError:
    print("[WARN] scikit-learn not installed")

print("[OK] Dependencies installed")

## Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    major, minor = torch.cuda.get_device_capability(0)
    device = 'cuda'
    torch.backends.cudnn.benchmark = True
    print(f"[GPU] {gpu_name} | {total_mem:.1f}GB | CUDA {torch.version.cuda} | Compute {major}.{minor}")
else:
    device = 'cpu'
    print("[WARN] No GPU available - training will be slow!")

In [ ]:
# GPU-specific configuration (auto-scaling)
GPU_CONFIGS = {
    'A100': {'batch': 512, 'hidden': 512, 'heads': 16, 'embed': 1024, 'lstm_layers': 3, 'ff_dim': 4096},
    'H100': {'batch': 512, 'hidden': 512, 'heads': 16, 'embed': 1024, 'lstm_layers': 3, 'ff_dim': 4096},
    'L4':   {'batch': 384, 'hidden': 384, 'heads': 12, 'embed': 768, 'lstm_layers': 2, 'ff_dim': 3072},
    'T4':   {'batch': 256, 'hidden': 256, 'heads': 8, 'embed': 512, 'lstm_layers': 2, 'ff_dim': 2048},
}

def detect_gpu():
    if not torch.cuda.is_available():
        return 'T4'  # Default
    name = torch.cuda.get_device_name(0).upper()
    for gpu in GPU_CONFIGS:
        if gpu in name:
            return gpu
    return 'T4'

GPU_TYPE = detect_gpu()
CONFIG = GPU_CONFIGS[GPU_TYPE]

print(f"\n{'='*60}")
print(f"CPC + REGRESSION - GPU OPTIMIZED")
print(f"{'='*60}")
print(f"Configured for: {GPU_TYPE}")
print(f"\nArchitecture:")
print(f"  Hidden dim: {CONFIG['hidden']}")
print(f"  Embed dim: {CONFIG['embed']}")
print(f"  LSTM layers: {CONFIG['lstm_layers']}")
print(f"  Attention heads: {CONFIG['heads']}")
print(f"  FF dim: {CONFIG['ff_dim']}")
print(f"\nTraining:")
print(f"  Batch size: {CONFIG['batch']}")
print(f"{'='*60}")

## Import Modules

In [ ]:
import os
import sys
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
src_path = f'/content/{repo_name}/ai_model/src'
ai_model_path = f'/content/{repo_name}/ai_model'

if src_path not in sys.path:
    sys.path.insert(0, src_path)

os.chdir(ai_model_path)
print(f"Working directory: {os.getcwd()}")

try:
    from cpc_regression import (
        CPCEncoder,
        CPCModel,
        ProbabilisticReturnHead,
        KellyPositionSizer,
        CPCConfig,
        RegressionConfig,
        KellyConfig,
    )
    from cpc_regression.cpc_model import CPCRegressionModel
    from cpc_regression.return_head import CalibrationMetrics, MultiTaskLoss
    from cpc_regression.kelly_sizer import KellyBacktester
    from training import train_cpc, train_regression
    from training.cpc_trainer import load_pretrained_encoder
    from training.regression_trainer import load_regression_model
    
    print("\n[OK] All CPC modules imported successfully!")
    print("\nAvailable components:")
    print("  - CPCEncoder: Bidirectional LSTM + Attention encoder")
    print("  - CPCModel: Contrastive Predictive Coding with InfoNCE")
    print("  - ProbabilisticReturnHead: Gaussian NLL return prediction")
    print("  - KellyPositionSizer: Optimal position sizing")
    print("  - train_cpc: Phase 1 pretraining")
    print("  - train_regression: Phase 2 fine-tuning")
    
except ImportError as e:
    print(f"[ERROR] Import failed: {e}")
    print("Make sure the cpc_regression module exists in ai_model/src/")

## Load and Verify Data

In [ ]:
# Set data directory
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
DATA_DIR = Path(f'/content/{repo_name}/ai_model/data/processed')
OUTPUT_DIR = Path(f'/content/{repo_name}/ai_model/models/cpc_regression')
LOG_DIR = OUTPUT_DIR / 'logs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

# Load preprocessed data
with open(DATA_DIR / 'train_samples.pkl', 'rb') as f:
    train_samples = pickle.load(f)

with open(DATA_DIR / 'val_samples.pkl', 'rb') as f:
    val_samples = pickle.load(f)

with open(DATA_DIR / 'test_samples.pkl', 'rb') as f:
    test_samples = pickle.load(f)

print(f"\n[DATA] Training samples: {len(train_samples):,}")
print(f"[DATA] Validation samples: {len(val_samples):,}")
print(f"[DATA] Test samples: {len(test_samples):,}")

# Verify sample structure
sample = train_samples[0]
print(f"\n[SAMPLE] Keys: {list(sample.keys())}")
print(f"[SAMPLE] Features shape: {sample['features'].shape}")

In [ ]:
# Analyze return and drawdown distributions
returns = [s['potential_profit_pct'] for s in train_samples]
drawdowns = [s['drawdown_pct'] for s in train_samples]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(returns, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(returns), color='red', linestyle='--', label=f'Mean: {np.mean(returns):.2%}')
axes[0].axvline(np.median(returns), color='blue', linestyle=':', label=f'Median: {np.median(returns):.2%}')
axes[0].set_xlabel('Potential Profit %')
axes[0].set_ylabel('Count')
axes[0].set_title('Return Distribution')
axes[0].legend()

axes[1].hist(drawdowns, bins=50, edgecolor='black', alpha=0.7, color='red')
axes[1].axvline(np.mean(drawdowns), color='blue', linestyle='--', label=f'Mean: {np.mean(drawdowns):.2%}')
axes[1].set_xlabel('Drawdown %')
axes[1].set_ylabel('Count')
axes[1].set_title('Drawdown Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nReturn stats: mean={np.mean(returns):.2%}, std={np.std(returns):.2%}, min={np.min(returns):.2%}, max={np.max(returns):.2%}")
print(f"Drawdown stats: mean={np.mean(drawdowns):.2%}, std={np.std(drawdowns):.2%}")
print(f"\nPositive returns: {sum(1 for r in returns if r > 0) / len(returns):.1%}")
print(f"Returns > 5%: {sum(1 for r in returns if r > 0.05) / len(returns):.1%}")

## Phase 1: CPC Pretraining

Self-supervised learning using **Contrastive Predictive Coding**.

The model learns to predict future embeddings from context using InfoNCE loss.
No labels are used - this is purely self-supervised.

In [ ]:
# Create CPC config with GPU-specific settings
cpc_config = CPCConfig(
    input_dim=14,
    hidden_dim=CONFIG['hidden'],
    embed_dim=CONFIG['embed'],
    lstm_layers=CONFIG['lstm_layers'],
    n_heads=CONFIG['heads'],
    ff_dim=CONFIG['ff_dim'],
    ar_hidden=CONFIG['hidden'],
    batch_size=CONFIG['batch'],
    learning_rate=1e-4,
    warmup_epochs=5,
    total_epochs=50,
    prediction_steps=[1, 2, 3, 5, 10],
    temperature=0.07,
    max_seq_len=128,
    min_seq_len=20,
)

print("="*60)
print("PHASE 1: CPC PRETRAINING CONFIGURATION")
print("="*60)
print(f"\nEncoder Architecture:")
print(f"  Input: 14 features")
print(f"  Hidden: {cpc_config.hidden_dim}")
print(f"  Embed: {cpc_config.embed_dim}")
print(f"  LSTM: {cpc_config.lstm_layers} layers (bidirectional)")
print(f"  Attention: {cpc_config.n_heads} heads")
print(f"  FF: {cpc_config.ff_dim}")
print(f"\nCPC Settings:")
print(f"  Prediction steps: {cpc_config.prediction_steps}")
print(f"  Temperature: {cpc_config.temperature}")
print(f"  AR hidden: {cpc_config.ar_hidden}")
print(f"\nTraining:")
print(f"  Batch size: {cpc_config.batch_size}")
print(f"  Learning rate: {cpc_config.learning_rate}")
print(f"  Warmup epochs: {cpc_config.warmup_epochs}")
print(f"  Total epochs: {cpc_config.total_epochs}")
print("="*60)

In [ ]:
# Run CPC pretraining
print("Starting CPC Pretraining...\n")

cpc_results = train_cpc(
    data_dir=str(DATA_DIR),
    output_dir=str(OUTPUT_DIR),
    config=cpc_config,
    device=device,
    verbose=1,
)

print(f"\n{'='*60}")
print("CPC PRETRAINING COMPLETE")
print(f"{'='*60}")
print(f"Best epoch: {cpc_results['best_epoch']}")
print(f"Best val loss: {cpc_results['best_val_loss']:.4f}")
print(f"Best val accuracy: {cpc_results['history']['val_acc'][cpc_results['best_epoch']-1]:.2%}")
print(f"Training time: {cpc_results['total_time_seconds']/60:.1f} minutes")
print(f"Model saved to: {OUTPUT_DIR / 'best_cpc_model.pt'}")

In [ ]:
# Plot CPC training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(cpc_results['history']['train_loss'], label='Train', alpha=0.8)
axes[0].plot(cpc_results['history']['val_loss'], label='Validation', alpha=0.8)
axes[0].axvline(cpc_results['best_epoch']-1, color='red', linestyle='--', alpha=0.5, label=f'Best: {cpc_results["best_epoch"]}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('CPC Loss (InfoNCE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(cpc_results['history']['train_acc'], label='Train', alpha=0.8)
axes[1].plot(cpc_results['history']['val_acc'], label='Validation', alpha=0.8)
axes[1].axvline(cpc_results['best_epoch']-1, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('CPC Prediction Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].plot(cpc_results['history']['lr'], color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Visualize Learned Representations (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

# Load best encoder
encoder, _ = load_pretrained_encoder(
    str(OUTPUT_DIR / 'best_cpc_model.pt'),
    device=device
)
encoder.eval()

# Extract embeddings for a subset of samples
N_SAMPLES = 2000
embeddings = []
return_values = []

print(f"Extracting embeddings for {N_SAMPLES} samples...")
with torch.no_grad():
    for sample in val_samples[:N_SAMPLES]:
        features = torch.FloatTensor(sample['features']).unsqueeze(0)
        if device == 'cuda':
            features = features.cuda()
        z = encoder.get_last_embedding(features)
        embeddings.append(z.cpu().numpy())
        return_values.append(sample['potential_profit_pct'])

embeddings = np.vstack(embeddings)
return_values = np.array(return_values)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Running t-SNE...")

# Run t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
embeddings_2d = tsne.fit_transform(embeddings)

# Plot colored by return
plt.figure(figsize=(12, 8))

# Clip extreme values for better visualization
returns_clipped = np.clip(return_values, -0.2, 0.5)
scatter = plt.scatter(
    embeddings_2d[:, 0], embeddings_2d[:, 1], 
    c=returns_clipped, cmap='RdYlGn', 
    alpha=0.6, s=15, vmin=-0.1, vmax=0.2
)
plt.colorbar(scatter, label='Return %')
plt.title('t-SNE of CPC Embeddings (colored by return)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

print(f"\nVisualization complete. Similar returns should cluster together.")

## Phase 2: Return Regression

Fine-tune the pretrained encoder for **return prediction with uncertainty**.

**Multi-task learning:**
- Primary: Expected return (mean + variance) using Gaussian NLL
- Auxiliary: Expected drawdown (MSE loss)

In [ ]:
# Create regression config
reg_config = RegressionConfig(
    hidden_dims=[256, 128],
    predict_drawdown=True,
    dropout=0.1,
    batch_size=CONFIG['batch'] // 2,  # Smaller batch for fine-tuning
    learning_rate=3e-5,
    encoder_lr_mult=0.1,  # Encoder learns 10x slower
    warmup_epochs=2,
    freeze_encoder_epochs=5,  # Freeze encoder first 5 epochs
    total_epochs=30,
    min_log_var=-6.0,
    var_reg_weight=0.1,
    drawdown_weight=0.3,
    cpc_regularization=0.1,
    patience=10,
)

print("="*60)
print("PHASE 2: REGRESSION CONFIGURATION")
print("="*60)
print(f"\nHead Architecture:")
print(f"  Hidden dims: {reg_config.hidden_dims}")
print(f"  Predict drawdown: {reg_config.predict_drawdown}")
print(f"  Dropout: {reg_config.dropout}")
print(f"\nTraining:")
print(f"  Batch size: {reg_config.batch_size}")
print(f"  Learning rate: {reg_config.learning_rate}")
print(f"  Encoder LR mult: {reg_config.encoder_lr_mult}")
print(f"  Freeze epochs: {reg_config.freeze_encoder_epochs}")
print(f"  Total epochs: {reg_config.total_epochs}")
print(f"\nLoss Settings:")
print(f"  Drawdown weight: {reg_config.drawdown_weight}")
print(f"  Min log variance: {reg_config.min_log_var}")
print(f"  Variance regularization: {reg_config.var_reg_weight}")
print("="*60)

In [ ]:
# Run regression training
print("Starting Regression Training...\n")

reg_results = train_regression(
    pretrained_encoder_path=str(OUTPUT_DIR / 'best_cpc_model.pt'),
    data_dir=str(DATA_DIR),
    output_dir=str(OUTPUT_DIR),
    cpc_config=cpc_config,
    reg_config=reg_config,
    device=device,
    verbose=1,
)

print(f"\n{'='*60}")
print("REGRESSION TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Best epoch: {reg_results['best_epoch']}")
print(f"Best val loss: {reg_results['best_val_loss']:.4f}")
print(f"Final MAE: {reg_results['final_val_mae']:.4f}")
print(f"Final correlation: {reg_results['final_correlation']:.3f}")
print(f"Calibration error: {reg_results['final_calibration_error']:.3f}")
print(f"Training time: {reg_results['total_time_seconds']/60:.1f} minutes")
print(f"Model saved to: {OUTPUT_DIR / 'best_regression_model.pt'}")

In [ ]:
# Plot regression training curves
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Loss
axes[0, 0].plot(reg_results['history']['train_loss'], label='Train', alpha=0.8)
axes[0, 0].plot(reg_results['history']['val_loss'], label='Validation', alpha=0.8)
axes[0, 0].axvline(reg_results['best_epoch']-1, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Gaussian NLL Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# MAE
axes[0, 1].plot(reg_results['history']['train_mae'], label='Train', alpha=0.8)
axes[0, 1].plot(reg_results['history']['val_mae'], label='Validation', alpha=0.8)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('MAE')
axes[0, 1].set_title('Mean Absolute Error')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Correlation
axes[1, 0].plot(reg_results['history']['val_correlation'], color='green', alpha=0.8)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Correlation')
axes[1, 0].set_title('Prediction Correlation')
axes[1, 0].grid(True, alpha=0.3)

# Calibration error
axes[1, 1].plot(reg_results['history']['calibration_error'], color='purple', alpha=0.8)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Calibration Error')
axes[1, 1].set_title('Uncertainty Calibration Error')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Calibration Analysis

A well-calibrated model should have:
- ~68% of predictions within 1 sigma
- ~95% of predictions within 2 sigma

In [ ]:
# Load best model and evaluate on test set
model, configs = load_regression_model(
    str(OUTPUT_DIR / 'best_regression_model.pt'),
    device=device
)

# Collect predictions
all_mus = []
all_log_vars = []
all_targets = []
all_dd_preds = []
all_dd_targets = []

print(f"Evaluating on {len(test_samples)} test samples...")
model.eval()
with torch.no_grad():
    for i, sample in enumerate(test_samples):
        features = torch.FloatTensor(sample['features']).unsqueeze(0)
        if device == 'cuda':
            features = features.cuda()
        
        outputs = model(features)
        mu, log_var, dd_pred = outputs
        
        all_mus.append(mu.cpu().item())
        all_log_vars.append(log_var.cpu().item())
        all_targets.append(sample['potential_profit_pct'])
        all_dd_preds.append(dd_pred.cpu().item())
        all_dd_targets.append(sample['drawdown_pct'])
        
        if (i + 1) % 2000 == 0:
            print(f"  Processed {i+1}/{len(test_samples)}")

all_mus = np.array(all_mus)
all_log_vars = np.array(all_log_vars)
all_targets = np.array(all_targets)
all_dd_preds = np.array(all_dd_preds)
all_dd_targets = np.array(all_dd_targets)

# Calibration metrics
cal = CalibrationMetrics.compute(
    torch.FloatTensor(all_mus),
    torch.FloatTensor(all_log_vars),
    torch.FloatTensor(all_targets)
)

print(f"\n{'='*60}")
print("CALIBRATION METRICS")
print(f"{'='*60}")
print(f"Within 1 sigma: {cal['within_1_sigma']:.1%} (expected: 68.3%)")
print(f"Within 2 sigma: {cal['within_2_sigma']:.1%} (expected: 95.4%)")
print(f"Within 3 sigma: {cal['within_3_sigma']:.1%} (expected: 99.7%)")
print(f"Calibration error: {cal['calibration_error']:.3f}")
print(f"Mean |Z-score|: {cal['mean_z_score']:.2f}")
print(f"\nPrediction correlation: {np.corrcoef(all_targets, all_mus)[0,1]:.3f}")
print(f"Drawdown correlation: {np.corrcoef(all_dd_targets, all_dd_preds)[0,1]:.3f}")

In [ ]:
# Plot predicted vs actual
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Return prediction scatter
axes[0].scatter(all_targets, all_mus, alpha=0.2, s=3)
lims = [-0.3, 0.5]
axes[0].plot(lims, lims, 'r--', label='Perfect', linewidth=2)
axes[0].set_xlabel('Actual Return')
axes[0].set_ylabel('Predicted Return')
axes[0].set_title(f'Return Prediction (corr={np.corrcoef(all_targets, all_mus)[0,1]:.3f})')
axes[0].legend()
axes[0].set_xlim(lims)
axes[0].set_ylim(lims)
axes[0].grid(True, alpha=0.3)

# Uncertainty calibration histogram
sigmas = np.exp(0.5 * all_log_vars)
z_scores = np.abs((all_targets - all_mus) / sigmas)

axes[1].hist(z_scores[z_scores < 5], bins=50, density=True, alpha=0.7, edgecolor='black')
# Expected half-normal distribution
from scipy import stats
x = np.linspace(0, 4, 100)
axes[1].plot(x, 2 * stats.norm.pdf(x), 'r-', label='Expected', linewidth=2)
axes[1].axvline(1, color='green', linestyle='--', alpha=0.7, label='1 sigma')
axes[1].axvline(2, color='orange', linestyle='--', alpha=0.7, label='2 sigma')
axes[1].set_xlabel('|Z-score|')
axes[1].set_ylabel('Density')
axes[1].set_title('Uncertainty Calibration')
axes[1].legend()
axes[1].set_xlim([0, 4])

# Variance distribution
axes[2].hist(sigmas, bins=50, alpha=0.7, edgecolor='black')
axes[2].set_xlabel('Predicted Sigma')
axes[2].set_ylabel('Count')
axes[2].set_title(f'Predicted Uncertainty (mean={np.mean(sigmas):.3f})')

plt.tight_layout()
plt.show()

## Kelly Backtesting

Test the trading strategy using Kelly Criterion position sizing.

**Kelly Formula:** `f* = mu / sigma^2`

We use **Quarter Kelly (0.25)** for conservative sizing.

In [ ]:
# Create Kelly sizer (Quarter Kelly - conservative)
kelly_config = KellyConfig(
    kelly_fraction=0.25,  # Quarter Kelly
    max_position=0.05,    # Max 5% per trade
    min_edge=0.02,        # Min 2% expected return
    max_variance=0.01,    # Max 10% variance
    transaction_cost=0.007,  # 0.7% fees
)

sizer = KellyPositionSizer(
    kelly_fraction=kelly_config.kelly_fraction,
    max_position=kelly_config.max_position,
    min_edge=kelly_config.min_edge,
    max_variance=kelly_config.max_variance,
    transaction_cost=kelly_config.transaction_cost,
)

# Run backtest
backtester = KellyBacktester(sizer, initial_capital=1.0)
backtest_results = backtester.run(
    mus=all_mus,
    log_vars=all_log_vars,
    actual_returns=all_targets,
)

print("="*60)
print("BACKTEST RESULTS (Quarter Kelly)")
print("="*60)
print(f"Final value: {backtest_results['final_value']:.4f}")
print(f"Total return: {backtest_results['total_return']:.2%}")
print(f"Total trades: {backtest_results['total_trades']}")
print(f"Win rate: {backtest_results['win_rate']:.1%}")
print(f"Avg trade return: {backtest_results['avg_trade_return']:.2%}")
print(f"Sharpe ratio: {backtest_results['sharpe_ratio']:.2f}")
print(f"Max drawdown: {backtest_results['max_drawdown']:.2%}")
print("="*60)

In [ ]:
# Compare different Kelly fractions
comparison = backtester.run_comparison(
    mus=all_mus,
    log_vars=all_log_vars,
    actual_returns=all_targets,
    kelly_fractions=[0.1, 0.25, 0.5, 1.0],
)

# Plot equity curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Equity curves
for name, results in comparison.items():
    axes[0].plot(results['equity_curve'], label=f"{name} ({results['total_return']:.1%})")

axes[0].set_xlabel('Trade')
axes[0].set_ylabel('Portfolio Value')
axes[0].set_title('Equity Curves (Different Kelly Fractions)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Summary bar chart
fractions = list(comparison.keys())
returns = [comparison[f]['total_return'] for f in fractions]
drawdowns = [comparison[f]['max_drawdown'] for f in fractions]

x = np.arange(len(fractions))
width = 0.35

axes[1].bar(x - width/2, [r*100 for r in returns], width, label='Return %', color='green', alpha=0.7)
axes[1].bar(x + width/2, [d*100 for d in drawdowns], width, label='Max DD %', color='red', alpha=0.7)
axes[1].set_xlabel('Kelly Fraction')
axes[1].set_ylabel('Percentage')
axes[1].set_title('Return vs Drawdown')
axes[1].set_xticks(x)
axes[1].set_xticklabels(fractions)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Summary table
print("\nKelly Fraction Comparison:")
print(f"{'Fraction':<15} {'Return':>10} {'Trades':>10} {'Win Rate':>10} {'Max DD':>10} {'Sharpe':>10}")
print("-" * 65)
for name, results in comparison.items():
    print(f"{name:<15} {results['total_return']:>9.1%} {results['total_trades']:>10} "
          f"{results['win_rate']:>9.1%} {results['max_drawdown']:>9.1%} {results['sharpe_ratio']:>10.2f}")

## TensorBoard Visualization

In [ ]:
# Load TensorBoard (if logs exist)
%load_ext tensorboard

repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
log_dir = f'/content/{repo_name}/ai_model/models/cpc_regression/logs'

import os
if os.path.exists(log_dir):
    %tensorboard --logdir {log_dir}
else:
    print(f"No TensorBoard logs found at {log_dir}")

## Save and Download Models

In [ ]:
# Save to Google Drive (if available)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    import shutil
    DRIVE_PATH = Path('/content/drive/MyDrive/gmgn_models/cpc_regression')
    DRIVE_PATH.mkdir(parents=True, exist_ok=True)
    
    # Copy models
    files_to_copy = [
        'best_cpc_model.pt',
        'best_regression_model.pt',
        'cpc_training_results.json',
        'regression_training_results.json',
    ]
    
    for fname in files_to_copy:
        src = OUTPUT_DIR / fname
        if src.exists():
            shutil.copy(src, DRIVE_PATH / fname)
            print(f"[SAVED] {fname}")
    
    print(f"\nModels saved to Google Drive: {DRIVE_PATH}")
except Exception as e:
    print(f"Google Drive not available: {e}")
    print(f"Models saved locally at: {OUTPUT_DIR}")

In [ ]:
# Download models to local machine
from google.colab import files

repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
model_dir = f'/content/{repo_name}/ai_model/models/cpc_regression'

# Download best regression model
model_file = f'{model_dir}/best_regression_model.pt'
if os.path.exists(model_file):
    files.download(model_file)
    print(f"[DOWNLOAD] {model_file}")

# Download CPC model
cpc_file = f'{model_dir}/best_cpc_model.pt'
if os.path.exists(cpc_file):
    files.download(cpc_file)
    print(f"[DOWNLOAD] {cpc_file}")

# Download results
results_file = f'{model_dir}/regression_training_results.json'
if os.path.exists(results_file):
    files.download(results_file)
    print(f"[DOWNLOAD] {results_file}")

print("\n[DONE] Files ready for download.")

## Training Summary

In [ ]:
# Print final summary
print("\n" + "="*70)
print("                     TRAINING COMPLETE")
print("="*70)

print(f"\n[GPU] {GPU_TYPE}")

print(f"\n[PHASE 1] CPC Pretraining")
print(f"  Best epoch: {cpc_results['best_epoch']}")
print(f"  Best val loss: {cpc_results['best_val_loss']:.4f}")
print(f"  Best val accuracy: {cpc_results['history']['val_acc'][cpc_results['best_epoch']-1]:.2%}")
print(f"  Training time: {cpc_results['total_time_seconds']/60:.1f} min")

print(f"\n[PHASE 2] Return Regression")
print(f"  Best epoch: {reg_results['best_epoch']}")
print(f"  Best val loss: {reg_results['best_val_loss']:.4f}")
print(f"  Final MAE: {reg_results['final_val_mae']:.4f}")
print(f"  Final correlation: {reg_results['final_correlation']:.3f}")
print(f"  Calibration error: {reg_results['final_calibration_error']:.3f}")
print(f"  Training time: {reg_results['total_time_seconds']/60:.1f} min")

print(f"\n[BACKTEST] Quarter Kelly (0.25)")
print(f"  Total return: {backtest_results['total_return']:.2%}")
print(f"  Total trades: {backtest_results['total_trades']}")
print(f"  Win rate: {backtest_results['win_rate']:.1%}")
print(f"  Sharpe ratio: {backtest_results['sharpe_ratio']:.2f}")
print(f"  Max drawdown: {backtest_results['max_drawdown']:.2%}")

print(f"\n[OUTPUT] {OUTPUT_DIR}")
print("="*70)